# معمل التجربة: الفرق بين Vector Store و Vector Database

أهلاً بيك في معمل فهارس وقواعد بيانات المتجهات.
الهدف هنا إننا نفهم ببساطة وبشكل عملي:
- إمتى أستخدم Vector Store بسيط في الذاكرة (زي FAISS أو Chroma Local)؟
- وإمتى أحتاج Vector Database سحابية متكاملة (زي Pinecone أو AstraDB)؟

تعال نجرب بنفسنا ونقيس سرعة البحث في الذاكرة.

### 1. تجهيز بيئة العمل والتضمينات المحلية
يلا نحمل المكتبات ونشغل نموذج التضمين المحلي المجاني بدون استهلاك أي باقة أو مفاتيح:

In [1]:
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import time

load_dotenv()

# هنستخدم نموذج all-MiniLM-L6-v2 بيشتغل محلياً وسريع جداً
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("نموذج التضمين جاهز للعمل!")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_22104\2834637890.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

نموذج التضمين جاهز للعمل!


### 2. تخزين وثائق تجريبية وقياس سرعة البحث
تعال نضيف شوية وثائق بتعبر عن أدوات مختلفة، ونشوف الـ Vector Store بيسترجعها في كام جزء من الثانية:

In [2]:
docs = [
    Document(page_content="LangChain إطار عمل لتسهيل بناء تطبيقات الذكاء الاصطناعي وربط النماذج.", metadata={"category": "ai", "tier": "free"}),
    Document(page_content="FAISS مكتبة ممتازة من ميتا للبحث السريع في المتجهات داخل الذاكرة.", metadata={"category": "database", "tier": "free"}),
    Document(page_content="Pinecone قاعدة بيانات سحابية متقدمة للمتجهات مصممة للأحمال الضخمة والشركات.", metadata={"category": "database", "tier": "enterprise"}),
    Document(page_content="Chroma قاعدة بيانات مفتوحة المصدر مناسبة للمشاريع السريعة والنماذج الأولية.", metadata={"category": "database", "tier": "free"}),
]

# هنبني الفهرس في الذاكرة
vector_store = FAISS.from_documents(docs, embeddings)

# يلا نقيس زمن البحث
query = "عايز حل سحابي ضخم يناسب الشركات"
start = time.perf_counter()
results = vector_store.similarity_search(query, k=2)
elapsed_ms = (time.perf_counter() - start) * 1000

print(f"تم البحث في: {elapsed_ms:.2f} ملي ثانية\n")
for i, doc in enumerate(results, start=1):
    print(f"النتيجة رقم {i}: {doc.page_content}")
    print(f"الفئة: {doc.metadata['category']} | الخطة: {doc.metadata['tier']}\n")

تم البحث في: 11.16 ملي ثانية

النتيجة رقم 1: FAISS مكتبة ممتازة من ميتا للبحث السريع في المتجهات داخل الذاكرة.
الفئة: database | الخطة: free

النتيجة رقم 2: LangChain إطار عمل لتسهيل بناء تطبيقات الذكاء الاصطناعي وربط النماذج.
الفئة: ai | الخطة: free



### 3. التصفية بالبيانات الوصفية (Metadata Filtering)
طب لو حابين نجيب بس الأدوات المجانية؟ تعال نجرب الفلترة:

In [3]:
filtered = vector_store.similarity_search(
    "أدوات إدارة متجهات",
    k=2,
    filter={"tier": "free"}
)

print("النتائج المجانية فقط:")
for doc in filtered:
    print(f"- {doc.page_content}")

النتائج المجانية فقط:
- FAISS مكتبة ممتازة من ميتا للبحث السريع في المتجهات داخل الذاكرة.
- LangChain إطار عمل لتسهيل بناء تطبيقات الذكاء الاصطناعي وربط النماذج.


### 4. جرب بنفسك!
تقدر هنا تغير السؤال أو تضيف وثائق جديدة وتشوف سرعة ودقة البحث بنفسك: